In [1]:
# =================================================================================================
# Raw data : This is Questionnaire data from NHANES 2003-2004
# Purpose : Creating unified raw CSV file from NHANES questionnaire CSV files.
# Specification : Each NHANES dataset has SEQN "Participant ID", which is the unique respondent ID.
# Files Used : ALQ_C.csv, BPQ_C.csv, DIQ_C.csv, HIQ_C.csv, HUQ_C.csv, SMQ_C.csv, SMQFAM_C.csv, WHQ_C.csv
# Data Dictionary : CDC NHANES 2003-2004 Questionnaire Data Dictionary
# Data Availability Matrix : Created in this notebook using SEQN presence flags
# Source Site : https://wwwn.cdc.gov/nchs/nhanes/search/datapage.aspx?Component=Questionnaire&CycleBeginYear=2003
# Developer Notes : The notebook was created for Windows desktop environment with Python3.
# File Creator : Neha Khairnar
# =================================================================================================

# 📦 Step 1: Imports
import pandas as pd
from pathlib import Path

# =================================================================================================
# 📂 Step 2: Paths below resolved for locally run datasets
# =================================================================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = BASE_DIR / "data" / "Datasets" / "NHANE_Continuous_DemographicsData_2013_2014_Questionnaire"

alcohol_file = DATA_DIR / "ALQ_H.csv"
bp_chol_file = DATA_DIR / "BPQ_H.csv"
diabetes_file = DATA_DIR / "DIQ_H.csv"
insurance_file = DATA_DIR / "HIQ_H.csv"
access_file = DATA_DIR / "HUQ_H.csv"
smoking_file = DATA_DIR / "SMQ_H.csv"
household_smokers_file = DATA_DIR / "SMQFAM_H.csv"
weight_history_file = DATA_DIR / "WHQ_H.csv"

files = {
    "alcohol_ALQ_C": alcohol_file,
    "blood_pressure_cholesterol_BPQ_C": bp_chol_file,
    "diabetes_DIQ_C": diabetes_file,
    "health_insurance_HIQ_C": insurance_file,
    "hospital_access_HUQ_C": access_file,
    "smoking_SMQ_C": smoking_file,
    "household_smokers_SMQFAM_C": household_smokers_file,
    "weight_history_WHQ_C": weight_history_file
}

# =================================================================================================
# 📥 Step 3: Load ALL Files
# =================================================================================================

dfs = {}

for name, path in files.items():
    df_temp = pd.read_csv(path)
    df_temp.columns = df_temp.columns.str.upper()
    
    # SEQN presence flag for troubleshooting
    df_temp[f"SEQN_in_{name}"] = df_temp["SEQN"].notna()
    
    dfs[name] = df_temp

# =================================================================================================
# 📊 Step 4: File Structure Summary
# =================================================================================================

summary = []

for name, df_temp in dfs.items():
    summary.append({
        "file": name,
        "rows": df_temp.shape[0],
        "columns": df_temp.shape[1],
        "column_names": list(df_temp.columns)
    })

summary_df = pd.DataFrame(summary)

print("📊 NHANES Questionnaire File Structure Summary:\n")
display(summary_df[["file", "rows", "columns", "column_names"]])

for i, row in summary_df.iterrows():
    print("\n" + "="*80)
    print("FILE:", row["file"])
    print("COLUMNS:", len(row["column_names"]))
    print(row["column_names"])

# =================================================================================================
# 🔗 Step 5: Merge ALL columns using FULL OUTER JOIN on SEQN
# =================================================================================================

df = dfs["alcohol_ALQ_C"]

for name in [
    "blood_pressure_cholesterol_BPQ_C",
    "diabetes_DIQ_C",
    "health_insurance_HIQ_C",
    "hospital_access_HUQ_C",
    "smoking_SMQ_C",
    "household_smokers_SMQFAM_C",
    "weight_history_WHQ_C"
]:
    df = df.merge(
        dfs[name],
        on="SEQN",
        how="outer",
        suffixes=("", f"_{name}")
    )

# =================================================================================================
# 🧠 Step 6: SEQN Presence Flags
# =================================================================================================

for name in files.keys():
    flag_col = f"SEQN_in_{name}"
    present_col = f"present_{name}"
    
    if flag_col in df.columns:
        df[present_col] = df[flag_col].fillna(False)

display(df.head(20))

print("Unique SEQN count:", df["SEQN"].nunique())
print("Final shape:", df.shape)

# =================================================================================================
# 🧠 Step 7: Rename Columns for Readability
# Naming convention: OriginalVariable_description
# =================================================================================================

rename_map_questionnaire = {
    # ID
    "SEQN": "SEQN",

    # Alcohol Use - ALQ_C
    "ALQ101": "ALQ101_had_at_least_12_alcohol_drinks_lifetime",
    "ALQ110": "ALQ110_had_at_least_12_alcohol_drinks_past_year",
    "ALQ120Q": "ALQ120Q_days_drank_alcohol_past_12_months_quantity",
    "ALQ120U": "ALQ120U_days_drank_alcohol_past_12_months_unit",
    "ALQ130": "ALQ130_average_drinks_per_drinking_day",
    "ALQ140Q": "ALQ140Q_days_had_5_or_more_drinks_quantity",
    "ALQ140U": "ALQ140U_days_had_5_or_more_drinks_unit",

    # Blood Pressure & Cholesterol - BPQ_C
    "BPQ020": "BPQ020_ever_told_high_blood_pressure",
    "BPQ030": "BPQ030_told_high_blood_pressure_2_or_more_times",
    "BPQ040A": "BPQ040A_taking_prescription_for_hypertension",
    "BPQ050A": "BPQ050A_now_taking_prescribed_hypertension_medicine",
    "BPQ060": "BPQ060_ever_had_blood_cholesterol_checked",
    "BPQ070": "BPQ070_when_cholesterol_checked",
    "BPQ080": "BPQ080_ever_told_high_cholesterol",
    "BPQ090D": "BPQ090D_told_to_take_prescription_for_cholesterol",

    # Diabetes - DIQ_C
    "DIQ010": "DIQ010_ever_told_diabetes",
    "DID040": "DID040_age_when_first_told_diabetes",
    "DIQ050": "DIQ050_now_taking_insulin",
    "DID060": "DID060_how_long_taking_insulin",
    "DIQ060U": "DIQ060U_insulin_duration_unit",
    "DIQ070": "DIQ070_take_diabetic_pills",
    "DIQ170": "DIQ170_ever_told_pre_diabetes",
    "DIQ172": "DIQ172_ever_told_health_risk_for_diabetes",

    # Health Insurance - HIQ_C
    "HIQ011": "HIQ011_covered_by_health_insurance",
    "HIQ031A": "HIQ031A_private_insurance",
    "HIQ031B": "HIQ031B_medicare",
    "HIQ031C": "HIQ031C_medi_gap",
    "HIQ031D": "HIQ031D_medicaid",
    "HIQ031E": "HIQ031E_schip",
    "HIQ031F": "HIQ031F_military_health_care",
    "HIQ031G": "HIQ031G_indian_health_service",
    "HIQ031H": "HIQ031H_state_sponsored_plan",
    "HIQ031I": "HIQ031I_other_government_insurance",
    "HIQ031J": "HIQ031J_single_service_plan",
    "HIQ031AA": "HIQ031AA_no_insurance_type",
    "HIQ260": "HIQ260_gap_in_health_insurance_past_year",

    # Hospital Utilization & Access to Care - HUQ_C
    "HUQ010": "HUQ010_general_health_condition",
    "HUQ030": "HUQ030_routine_place_to_go_for_healthcare",
    "HUQ040": "HUQ040_type_place_most_often_go_for_healthcare",
    "HUQ050": "HUQ050_times_received_healthcare_past_year",
    "HUQ060": "HUQ060_overnight_hospital_patient_past_year",
    "HUQ070": "HUQ070_time_since_last_healthcare_visit",

    # Smoking Adult - SMQ_C
    "SMQ020": "SMQ020_smoked_at_least_100_cigarettes_lifetime",
    "SMD030": "SMD030_age_started_smoking_regularly",
    "SMQ040": "SMQ040_now_smoke_cigarettes",
    "SMQ050Q": "SMQ050Q_time_since_quit_smoking_quantity",
    "SMQ050U": "SMQ050U_time_since_quit_smoking_unit",
    "SMD070": "SMD070_cigarettes_smoked_per_day_when_smoked",
    "SMD075": "SMD075_cigarettes_smoked_per_day_now",

    # Household Smokers - SMQFAM_C
    "SMAQUEX": "SMAQUEX_smoking_family_questionnaire_status",
    "SMD410": "SMD410_number_of_smokers_in_home",
    "SMD415": "SMD415_number_of_people_who_smoke_inside_home",

    # Weight History - WHQ_C
    "WHD010": "WHD010_current_self_reported_height_inches",
    "WHD020": "WHD020_current_self_reported_weight_pounds",
    "WHQ030": "WHQ030_how_do_you_consider_your_weight",
    "WHQ040": "WHQ040_like_to_weigh_more_less_same",
    "WHD050": "WHD050_self_reported_weight_1_year_ago_pounds",
    "WHQ060": "WHQ060_weight_change_intentional",
    "WHQ070": "WHQ070_tried_to_lose_weight_past_year",
    "WHD080A": "WHD080A_weight_loss_method_ate_less_food",
    "WHD080B": "WHD080B_weight_loss_method_switched_foods",
    "WHD080C": "WHD080C_weight_loss_method_exercise",
    "WHD110": "WHD110_self_reported_weight_10_years_ago_pounds"
}

df = df.rename(columns={k: v for k, v in rename_map_questionnaire.items() if k in df.columns})

# =================================================================================================
# 👀 Step 8: Validate Final Result
# =================================================================================================

print("📋 Final Columns After Merge + Rename:")
print(df.columns.tolist())

print("\n📊 Final Shape:", df.shape)
print("Unique SEQN count:", df["SEQN"].nunique())

display(df.head(20))

# =================================================================================================
# 📊 Step 9: Data Availability Matrix
# =================================================================================================

availability_cols = [col for col in df.columns if col.startswith("present_")]

availability_matrix = df[["SEQN"] + availability_cols].copy()

print("📊 Data Availability Matrix:")
display(availability_matrix.head(20))

availability_summary = []

for col in availability_cols:
    availability_summary.append({
        "dataset": col.replace("present_", ""),
        "available_records": df[col].sum(),
        "missing_records": df.shape[0] - df[col].sum(),
        "availability_percent": round((df[col].sum() / df.shape[0]) * 100, 2)
    })

availability_summary_df = pd.DataFrame(availability_summary)

display(availability_summary_df)

# =================================================================================================
# 💾 Step 10: Save Final Output
# =================================================================================================

OUTPUT_DIR = BASE_DIR / "data" / "processed"/ "NHANE_Questionnair"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "neighbourhood_nhanes_questionnaire_2013_2014.csv"

df.to_csv(OUTPUT_PATH, index=False)

print("\n✅ Saved to:")
print(OUTPUT_PATH)

📊 NHANES Questionnaire File Structure Summary:



,file,rows,columns,column_names
0,alcohol_ALQ_C,5924,11,"[SEQN, ALQ101, ALQ110, ALQ120Q, ALQ120U, ALQ13..."
1,blood_pressure_cholesterol_BPQ_C,6464,15,"[SEQN, BPQ020, BPQ030, BPD035, BPQ040A, BPQ050..."
2,diabetes_DIQ_C,9770,55,"[SEQN, DIQ010, DID040, DIQ160, DIQ170, DIQ172,..."
3,health_insurance_HIQ_C,10175,18,"[SEQN, HIQ011, HIQ031A, HIQ031B, HIQ031C, HIQ0..."
4,hospital_access_HUQ_C,10175,11,"[SEQN, HUQ010, HUQ020, HUQ030, HUQ041, HUQ051,..."
5,smoking_SMQ_C,7168,33,"[SEQN, SMQ020, SMD030, SMQ040, SMQ050Q, SMQ050..."
6,household_smokers_SMQFAM_C,10175,5,"[SEQN, SMD460, SMD470, SMD480, SEQN_in_househo..."
7,weight_history_WHQ_C,6464,35,"[SEQN, WHD010, WHD020, WHQ030, WHQ040, WHD050,..."



FILE: alcohol_ALQ_C
COLUMNS: 11
['SEQN', 'ALQ101', 'ALQ110', 'ALQ120Q', 'ALQ120U', 'ALQ130', 'ALQ141Q', 'ALQ141U', 'ALQ151', 'ALQ160', 'SEQN_in_alcohol_ALQ_C']

FILE: blood_pressure_cholesterol_BPQ_C
COLUMNS: 15
['SEQN', 'BPQ020', 'BPQ030', 'BPD035', 'BPQ040A', 'BPQ050A', 'BPQ056', 'BPD058', 'BPQ059', 'BPQ080', 'BPQ060', 'BPQ070', 'BPQ090D', 'BPQ100D', 'SEQN_in_blood_pressure_cholesterol_BPQ_C']

FILE: diabetes_DIQ_C
COLUMNS: 55
['SEQN', 'DIQ010', 'DID040', 'DIQ160', 'DIQ170', 'DIQ172', 'DIQ175A', 'DIQ175B', 'DIQ175C', 'DIQ175D', 'DIQ175E', 'DIQ175F', 'DIQ175G', 'DIQ175H', 'DIQ175I', 'DIQ175J', 'DIQ175K', 'DIQ175L', 'DIQ175M', 'DIQ175N', 'DIQ175O', 'DIQ175P', 'DIQ175Q', 'DIQ175R', 'DIQ175S', 'DIQ175T', 'DIQ175U', 'DIQ175V', 'DIQ175W', 'DIQ175X', 'DIQ180', 'DIQ050', 'DID060', 'DIQ060U', 'DIQ070', 'DIQ230', 'DIQ240', 'DID250', 'DID260', 'DIQ260U', 'DIQ275', 'DIQ280', 'DIQ291', 'DIQ300S', 'DIQ300D', 'DID310S', 'DID310D', 'DID320', 'DID330', 'DID341', 'DID350', 'DIQ350U', 'DIQ360', 'DIQ08

C:\Users\NVK00\AppData\Local\Temp\ipykernel_25492\1669576018.py:116: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[present_col] = df[flag_col].fillna(False)
C:\Users\NVK00\AppData\Local\Temp\ipykernel_25492\1669576018.py:116: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[present_col] = df[flag_col].fillna(False)
C:\Users\NVK00\AppData\Local\Temp\ipykernel_25492\1669576018.py:116: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.

,SEQN,ALQ101,ALQ110,ALQ120Q,ALQ120U,ALQ130,ALQ141Q,ALQ141U,ALQ151,ALQ160,...,WHQ150,SEQN_in_weight_history_WHQ_C,present_alcohol_ALQ_C,present_blood_pressure_cholesterol_BPQ_C,present_diabetes_DIQ_C,present_health_insurance_HIQ_C,present_hospital_access_HUQ_C,present_smoking_SMQ_C,present_household_smokers_SMQFAM_C,present_weight_history_WHQ_C
0,73557.0,1.0,NaN,1.000000e+00,3.0,1.0,5.397605e-79,NaN,1.0,NaN,...,62.0,True,True,True,True,True,True,True,True,True
1,73558.0,1.0,NaN,7.000000e+00,1.0,4.0,2.000000e+00,1.0,1.0,5.397605e-79,...,25.0,True,True,True,True,True,True,True,True,True
2,73559.0,1.0,NaN,5.397605e-79,NaN,NaN,NaN,NaN,2.0,NaN,...,35.0,True,True,True,True,True,True,True,True,True
3,73560.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,False,False,True,True,True,False,True,False
4,73561.0,1.0,NaN,5.397605e-79,NaN,NaN,NaN,NaN,2.0,NaN,...,60.0,True,True,True,True,True,True,True,True,True
5,73562.0,1.0,NaN,5.000000e+00,3.0,1.0,5.397605e-79,NaN,2.0,5.397605e-79,...,54.0,True,True,True,True,True,True,True,True,True
6,73563.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,False,False,False,True,True,False,True,False
7,73564.0,2.0,1.0,2.000000e+00,3.0,1.0,5.397605e-79,NaN,2.0,NaN,...,61.0,True,True,True,True,True,True,True,True,True
8,73565.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,37.0,True,False,True,True,True,True,True,True,True
9,73566.0,1.0,NaN,1.000000e+00,1.0,1.0,5.397605e-79,NaN,2.0,NaN,...,45.0,True,True,True,True,True,True,True,True,True


...

,SEQN,ALQ101_had_at_least_12_alcohol_drinks_lifetime,ALQ110_had_at_least_12_alcohol_drinks_past_year,ALQ120Q_days_drank_alcohol_past_12_months_quantity,ALQ120U_days_drank_alcohol_past_12_months_unit,ALQ130_average_drinks_per_drinking_day,ALQ141Q,ALQ141U,ALQ151,ALQ160,...,WHQ150,SEQN_in_weight_history_WHQ_C,present_alcohol_ALQ_C,present_blood_pressure_cholesterol_BPQ_C,present_diabetes_DIQ_C,present_health_insurance_HIQ_C,present_hospital_access_HUQ_C,present_smoking_SMQ_C,present_household_smokers_SMQFAM_C,present_weight_history_WHQ_C
0,73557.0,1.0,NaN,1.000000e+00,3.0,1.0,5.397605e-79,NaN,1.0,NaN,...,62.0,True,True,True,True,True,True,True,True,True
1,73558.0,1.0,NaN,7.000000e+00,1.0,4.0,2.000000e+00,1.0,1.0,5.397605e-79,...,25.0,True,True,True,True,True,True,True,True,True
2,73559.0,1.0,NaN,5.397605e-79,NaN,NaN,NaN,NaN,2.0,NaN,...,35.0,True,True,True,True,True,True,True,True,True
3,73560.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,False,False,True,True,True,False,True,False
4,73561.0,1.0,NaN,5.397605e-79,NaN,NaN,NaN,NaN,2.0,NaN,...,60.0,True,True,True,True,True,True,True,True,True
5,73562.0,1.0,NaN,5.000000e+00,3.0,1.0,5.397605e-79,NaN,2.0,5.397605e-79,...,54.0,True,True,True,True,True,True,True,True,True
6,73563.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,False,False,False,True,True,False,True,False
7,73564.0,2.0,1.0,2.000000e+00,3.0,1.0,5.397605e-79,NaN,2.0,NaN,...,61.0,True,True,True,True,True,True,True,True,True
8,73565.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,37.0,True,False,True,True,True,True,True,True,True
9,73566.0,1.0,NaN,1.000000e+00,1.0,1.0,5.397605e-79,NaN,2.0,NaN,...,45.0,True,True,True,True,True,True,True,True,True


📊 Data Availability Matrix:


,SEQN,present_alcohol_ALQ_C,present_blood_pressure_cholesterol_BPQ_C,present_diabetes_DIQ_C,present_health_insurance_HIQ_C,present_hospital_access_HUQ_C,present_smoking_SMQ_C,present_household_smokers_SMQFAM_C,present_weight_history_WHQ_C
0,73557.0,True,True,True,True,True,True,True,True
1,73558.0,True,True,True,True,True,True,True,True
2,73559.0,True,True,True,True,True,True,True,True
3,73560.0,False,False,True,True,True,False,True,False
4,73561.0,True,True,True,True,True,True,True,True
5,73562.0,True,True,True,True,True,True,True,True
6,73563.0,False,False,False,True,True,False,True,False
7,73564.0,True,True,True,True,True,True,True,True
8,73565.0,False,True,True,True,True,True,True,True
9,73566.0,True,True,True,True,True,True,True,True


,dataset,available_records,missing_records,availability_percent
0,alcohol_ALQ_C,5924,4251,58.22
1,blood_pressure_cholesterol_BPQ_C,6464,3711,63.53
2,diabetes_DIQ_C,9770,405,96.02
3,health_insurance_HIQ_C,10175,0,100.00
4,hospital_access_HUQ_C,10175,0,100.00
5,smoking_SMQ_C,7168,3007,70.45
6,household_smokers_SMQFAM_C,10175,0,100.00
7,weight_history_WHQ_C,6464,3711,63.53



✅ Saved to:
C:\Users\NVK00\Desktop\MyData\Projects\Clinical_Neighbourhood_ML_Data_Predictor\data\processed\NHANE_Questionnair\neighbourhood_nhanes_questionnaire_2013_2014.csv
